In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import VotingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import seaborn as sns
import matplotlib.pyplot as plt

# Baca data dan siapin kolom
data = pd.read_csv('Klasifikasi Katarak.csv', header=None, decimal=',')
data.columns = ['Feature1', 'Feature2', 'Feature3', 'Label']

# Pisahkan fitur dan label
X = data[['Feature1', 'Feature2', 'Feature3']].values
y = data['Label'].values

# Encode label biar bentuknya angka
le = LabelEncoder()
y = le.fit_transform(y)

# Standardisasi fitur (biar skala sama)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Tambahin fitur polinomial biar model bisa nangkap hubungan kompleks
poly = PolynomialFeatures(degree=2, interaction_only=False)
X = poly.fit_transform(X)

# Atasi masalah data nggak seimbang pake SMOTE
smote = SMOTE(random_state=42)
X, y = smote.fit_resample(X, y)

# Definisi model-modelnya
bpnn = MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42)
svm = make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))

# Voting Classifier (kombinasi model)
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')),
        ('bpnn', bpnn),
        ('svm', svm)
    ],
    voting='soft'  # Soft voting untuk rata-rata probabilitas
)

# Setup Stratified K-Fold buat validasi silang
skf = StratifiedKFold(n_splits=8, shuffle=True, random_state=42)
accuracies = []
recall_list = []
specificity_list = []
classification_reports = []
confusion_matrices = []

# Variabel buat simpan hasil fold terbaik
best_fold_accuracy = 0
best_fold_report = None
best_fold_cm = None

# Loop validasi silang
for fold, (train_index, test_index) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Latih model Voting Classifier
    voting_clf.fit(X_train, y_train)

    # Prediksi
    y_pred = voting_clf.predict(X_test)

    # Hitung akurasi
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

    # Simpan hasil fold terbaik
    if accuracy > best_fold_accuracy:
        best_fold_accuracy = accuracy
        best_fold_report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)
        best_fold_cm = confusion_matrix(y_test, y_pred)

    # Sensitivitas (recall) tiap kelas
    recall = recall_score(y_test, y_pred, average=None)
    recall_list.append(recall)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    confusion_matrices.append(cm)

    # Hitung spesifisitas tiap kelas
    specificity = []
    for i in range(len(le.classes_)):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        specificity.append(tn / (tn + fp))
    specificity_list.append(specificity)

    # Simpan laporan klasifikasi
    report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)
    classification_reports.append(report)

    print(f"Fold {fold} - Akurasi: {accuracy * 100:.2f}%")

# Hasil rata-rata validasi silang
mean_accuracy = np.mean(accuracies)
print(f"\nRata-rata Akurasi Cross-Validation: {mean_accuracy * 100:.2f}%")

# Rata-rata recall dan spesifisitas
mean_recall = np.mean(recall_list, axis=0)
mean_specificity = np.mean(specificity_list, axis=0)

for idx, label in enumerate(le.classes_):
    print(f"{label} - Rata-rata Sensitivitas: {mean_recall[idx] * 100:.2f}%, Rata-rata Spesifisitas: {mean_specificity[idx] * 100:.2f}%")

# Kombinasi laporan klasifikasi rata-rata
final_report = {}
for class_label in le.classes_:
    final_report[class_label] = {
        metric: np.mean([fold[class_label][metric] for fold in classification_reports])
        for metric in classification_reports[0][class_label]
    }

# Rata-rata akurasi
final_report['accuracy'] = np.mean([fold['accuracy'] for fold in classification_reports])

# Global metrics (macro avg & weighted avg)
for global_metric in ['macro avg', 'weighted avg']:
    final_report[global_metric] = {
        metric: np.mean([fold[global_metric][metric] for fold in classification_reports])
        for metric in classification_reports[0][global_metric]
    }

print("\nRata-rata Laporan Klasifikasi:")
print(pd.DataFrame(final_report).transpose())

# Tampilkan hasil fold terbaik
print(f"\nFold Terbaik - Akurasi: {best_fold_accuracy * 100:.2f}%")
print("\nFold Terbaik - Laporan Klasifikasi:")
print(pd.DataFrame(best_fold_report).transpose())

plt.figure(figsize=(8, 6))
sns.heatmap(best_fold_cm, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title(f'Confusion Matrix Fold Terbaik (Akurasi: {best_fold_accuracy * 100:.2f}%)')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.tight_layout()
plt.show()